In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import chi2, norm
from dataclasses import asdict
from joblib import Parallel, delayed
from datetime import datetime

from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigsh

from core import (
    Config, Simulator, NumpyEncoder,
    save_results, load_results,
    get_model_description, create_shared_experiment,
    build_sim_kwargs, compute_beta_from_ratio,
    setup_plot_style, STRATEGY_STYLES, XLABEL_LOG_RATIO,
)


def sample_L_pa(p, a, K=20000, oversample=40):

    out = []
    while len(out) < K:
        D = np.random.standard_normal((K * oversample, p))
        keep = (D ** 2).sum(axis=1) <= a
        out.extend(D[keep, 0].tolist())
    return np.asarray(out[:K])


def compute_qbar(p, a, v_p, alpha, K=20000):


    q_level = 1 - alpha / 2
    E0 = np.random.standard_normal(K)
    La = sample_L_pa(p, a, K)
    best = 0.0
    for R2 in np.linspace(0.0, 1.0, 41):
        omega = float(np.quantile(np.sqrt(1 - R2) * E0 + np.sqrt(R2) * La, q_level))
        denom = np.sqrt(max(1 - (1 - v_p) * R2, 1e-12))
        best = max(best, omega / denom)
    return best


def compute_lambda_max(adj):

    adj_sp = csr_matrix(adj)
    adj2 = adj_sp @ adj_sp
    A_dep = ((adj_sp + adj2) > 0).astype(float)
    A_dep.setdiag(0)
    A_dep.eliminate_zeros()
    vals = eigsh(A_dep.astype(float), k=1, which='LM', return_eigenvectors=False)
    return float(vals[0])


def compute_local_dep_degrees(adj):

    adj_sp = csr_matrix(adj)
    adj2 = adj_sp @ adj_sp
    A_dep = ((adj_sp + adj2) > 0).astype(float)
    A_dep.setdiag(1)
    A_dep.eliminate_zeros()
    return np.asarray(A_dep.sum(axis=1)).flatten()


def compute_U_hat(Z, Y, X_rerand, d_local, p_treat, n):

    phi = np.hstack([Y.reshape(-1, 1), X_rerand])
    treated = (Z == 1)
    control = (Z == 0)
    phi_hat = phi.copy()
    if treated.sum() > 0:
        phi_hat[treated] -= phi[treated].mean(axis=0)
    if control.sum() > 0:
        phi_hat[control] -= phi[control].mean(axis=0)
    psi_sq = np.where(Z == 1, 1.0 / p_treat**2, 1.0 / (1 - p_treat)**2)
    scale = np.sqrt(d_local * psi_sq)
    weighted = phi_hat * scale[:, np.newaxis]
    return (1.0 / n**2) * (weighted.T @ weighted)


def compute_v_hat_n(U_hat, Sigma_xx, v_p, reg=1e-8, n_bisect=60):


    U11 = U_hat[0, 0]
    U12 = U_hat[0, 1:]
    U22 = U_hat[1:, 1:]
    p = U22.shape[0]


    diff = U22 - Sigma_xx
    eigs_d = np.linalg.eigvalsh(diff)
    if eigs_d.min() < reg:
        diff = diff + (abs(eigs_d.min()) + reg) * np.eye(p)
    Sxx = Sigma_xx.copy()
    eigs_s = np.linalg.eigvalsh(Sxx)
    if eigs_s.min() < reg:
        Sxx = Sxx + (abs(eigs_s.min()) + reg) * np.eye(p)

    def _M(t):
        return Sxx + t * diff

    def _G(t):
        Mi = np.linalg.inv(_M(t))
        inner = Sxx + (t ** 2) * diff
        return float(U12 @ Mi @ inner @ Mi @ U12)

    def _v_at(t):
        Mi = np.linalg.inv(_M(t))
        inner = (1.0 - v_p) * Sxx + (t ** 2) * diff
        return float(U11 - U12 @ Mi @ inner @ Mi @ U12)

    t_lo = 1.0 - v_p
    if _G(t_lo) <= U11:


        Mi = np.linalg.inv(_M(t_lo))
        return float(U11 - (1.0 - v_p) * (U12 @ Mi @ U12))


    lo, hi = t_lo, 1.0
    for _ in range(n_bisect):
        mid = 0.5 * (lo + hi)
        if _G(mid) > U11:
            lo = mid
        else:
            hi = mid
    return _v_at(hi)


def compute_U_hat_savje(Z, Y, X_rerand, lambda_max, p_treat, n):


    phi = np.hstack([Y.reshape(-1, 1), X_rerand])
    treated = (Z == 1)
    control = (Z == 0)
    phi_hat = phi.copy()
    if treated.sum() > 0:
        phi_hat[treated] -= phi[treated].mean(axis=0)
    if control.sum() > 0:
        phi_hat[control] -= phi[control].mean(axis=0)
    psi_sq = np.where(Z == 1, 1.0 / p_treat**2, 1.0 / (1 - p_treat)**2)
    scale = np.sqrt(psi_sq)
    weighted = phi_hat * scale[:, np.newaxis]
    return (lambda_max / n**2) * (weighted.T @ weighted)


def compute_U_hat_uncen(Z, Y, X_rerand, d_local, p_treat, n):


    phi = np.hstack([Y.reshape(-1, 1), X_rerand])

    psi_sq = np.where(Z == 1, 1.0 / p_treat**2, 1.0 / (1 - p_treat)**2)
    scale = np.sqrt(d_local * psi_sq)
    weighted = phi * scale[:, np.newaxis]
    return (1.0 / n**2) * (weighted.T @ weighted)


def _run_single_cv(cfg, shared, total_signal_std, target_delta, seed):

    np.random.seed(seed)
    std_pre, std_trt = shared['std_pre'], shared['std_trt']
    beta_pre_val, beta_trt_val = compute_beta_from_ratio(target_delta, std_pre, std_trt, total_signal_std)

    actual_ratio = (beta_trt_val * std_trt) / (beta_pre_val * std_pre) if beta_pre_val * std_pre > 0 else float('inf')
    actual_log_ratio = np.log2(actual_ratio) if actual_ratio > 0 and np.isfinite(actual_ratio) else target_delta

    sim = Simulator(**build_sim_kwargs(cfg, shared, beta_pre_val, beta_trt_val))
    d_local = compute_local_dep_degrees(sim.adj)
    strategies = ['static', 'dynamic', 'all']


    tau_y_list = []
    tau_x_dict = {s: [] for s in strategies}
    u_n_bern_list = []


    sqrt_u_bern_list = []

    for _ in range(cfg.n_mc):
        Z = np.random.binomial(1, cfg.p_treat, sim.n)
        if not (0 < Z.sum() < sim.n): continue
        Y = sim._outcome(Z)
        tau_y_list.append(sim._hajek(Z, Y)[0])

        X_dyn = sim._get_covs(Z, cfg.rerand_covs)
        tau_x_dict['static'].append(sim._hajek(Z, sim.X_pre))
        tau_x_dict['dynamic'].append(sim._hajek(Z, X_dyn))
        tau_x_dict['all'].append(sim._hajek(Z, np.hstack([sim.X_pre, X_dyn])))


        U_hat_b = compute_U_hat(Z, Y, sim.X_pre, d_local, cfg.p_treat, sim.n)
        u_n_bern_list.append(float(U_hat_b[0, 0]))
        sqrt_u_bern_list.append(float(np.sqrt(max(float(U_hat_b[0, 0]), 0.0))))

    mean_u_hat_n_bern_global = float(np.mean(u_n_bern_list)) if u_n_bern_list else float('nan')

    mean_sqrt_u_bern_global = float(np.mean(sqrt_u_bern_list)) if sqrt_u_bern_list else float('nan')

    tau_y_arr = np.array(tau_y_list)
    true_var_tau = np.var(tau_y_arr)

    strat_data = {}
    for s in strategies:
        arr = np.array(tau_x_dict[s])
        if arr.ndim == 1: arr = arr.reshape(-1, 1)
        Sigma_xx = np.cov(arr, rowvar=False)
        if Sigma_xx.ndim == 0: Sigma_xx = np.array([[float(Sigma_xx)]])
        p_cov = arr.shape[1]
        Sigma_xx_inv = np.linalg.inv(Sigma_xx + 1e-8 * np.eye(p_cov))
        thresh = chi2.ppf(cfg.acceptance_rate, p_cov)
        v_p = chi2.cdf(thresh, p_cov + 2) / chi2.cdf(thresh, p_cov)

        n_valid = min(len(tau_y_arr), arr.shape[0])
        joint = np.column_stack([tau_y_arr[:n_valid].reshape(-1, 1), arr[:n_valid]])
        C_joint = np.cov(joint, rowvar=False)
        cov_y_x = C_joint[0, 1:]
        R_sq = float(cov_y_x @ Sigma_xx_inv @ cov_y_x / true_var_tau) if true_var_tau > 0 else 0.0
        R_sq = min(max(R_sq, 0.0), 1.0)

        strat_data[s] = {
            'Sigma_xx': Sigma_xx, 'Sigma_xx_inv': Sigma_xx_inv,
            'p_cov': p_cov, 'thresh': thresh, 'v_p': v_p, 'R_squared': R_sq,

            'q_bar': compute_qbar(p_cov, thresh, v_p, cfg.ci_alpha),
        }


    lambda_max_val = compute_lambda_max(sim.adj)
    z_crit = norm.ppf(1 - cfg.ci_alpha / 2)
    tau_true = cfg.beta_treatment

    res = {s: {'tau_hat': [], 'u_hat_n': [], 'v_hat_n': [], 'v_hat_n_savje': [], 'v_hat_n_uncen': [],
               'L_rem': [],
               'ci_covers_uncond': [], 'ci_covers_cond': []} for s in strategies}

    n_needed = cfg.n_accepted
    for _ in range(n_needed * 200):
        if all(len(res[s]['tau_hat']) >= n_needed for s in strategies): break
        Z = np.random.binomial(1, cfg.p_treat, sim.n)
        if not (0 < Z.sum() < sim.n): continue

        X_dyn = sim._get_covs(Z, cfg.rerand_covs)
        X_dict = {'static': sim.X_pre, 'dynamic': X_dyn, 'all': np.hstack([sim.X_pre, X_dyn])}
        Y = None

        for s in strategies:
            if len(res[s]['tau_hat']) >= n_needed: continue
            sd = strat_data[s]
            tau_x = sim._hajek(Z, X_dict[s])
            if tau_x @ sd['Sigma_xx_inv'] @ tau_x > sd['thresh']: continue

            if Y is None: Y = sim._outcome(Z)
            tau_y = sim._hajek(Z, Y)[0]

            U_hat = compute_U_hat(Z, Y, X_dict[s], d_local, cfg.p_treat, sim.n)
            u_n = U_hat[0, 0]
            try:
                v_n = max(compute_v_hat_n(U_hat, sd['Sigma_xx'], sd['v_p']), 0)
            except Exception:
                v_n = u_n


            v_n_savje = np.nan
            if s == 'all':
                try:
                    U_hat_savje = compute_U_hat_savje(Z, Y, X_dict[s], lambda_max_val, cfg.p_treat, sim.n)
                    v_n_savje = max(compute_v_hat_n(U_hat_savje, sd['Sigma_xx'], sd['v_p']), 0)
                except Exception:
                    v_n_savje = U_hat_savje[0, 0] if 'U_hat_savje' in dir() else np.nan


            v_n_uncen = np.nan
            if s == 'all':
                try:
                    U_hat_uncen = compute_U_hat_uncen(Z, Y, X_dict[s], d_local, cfg.p_treat, sim.n)
                    v_n_uncen = max(compute_v_hat_n(U_hat_uncen, sd['Sigma_xx'], sd['v_p']), 0)
                except Exception:
                    v_n_uncen = np.nan

            res[s]['tau_hat'].append(tau_y)
            res[s]['u_hat_n'].append(u_n)
            res[s]['v_hat_n'].append(v_n)
            res[s]['v_hat_n_savje'].append(v_n_savje)
            res[s]['v_hat_n_uncen'].append(v_n_uncen)


            L_rem = sd['q_bar'] * np.sqrt(max(v_n, 0.0))
            res[s]['L_rem'].append(float(L_rem))

            res[s]['ci_covers_uncond'].append(abs(tau_y - tau_true) <= z_crit * np.sqrt(max(u_n, 0)))

            res[s]['ci_covers_cond'].append(abs(tau_y - tau_true) <= sd['q_bar'] * np.sqrt(max(v_n, 0)))


    lambda_max = compute_lambda_max(sim.adj)
    summary = {
        'd_local_mean': float(np.mean(d_local)),
        'd_local_max': float(np.max(d_local)),
        'lambda_max': float(lambda_max),
        'target_log_ratio': float(target_delta),
        'actual_log_ratio': float(actual_log_ratio),
        'beta_pre': float(beta_pre_val), 'beta_trt': float(beta_trt_val),
    }
    for s in strategies:
        r = res[s]
        sd = strat_data[s]
        n_acc = len(r['tau_hat'])
        if n_acc == 0:
            summary[s] = {'n_accepted': 0}
            continue
        u_arr = np.array(r['u_hat_n'])
        v_arr = np.array(r['v_hat_n'])
        true_var_cond = np.var(r['tau_hat'])
        valid = u_arr > 0
        ratios = v_arr[valid] / u_arr[valid] if valid.sum() > 0 else np.array([1.0])

        L_rem_arr = np.array(r['L_rem'])
        mean_L_rem = float(np.mean(L_rem_arr)) if L_rem_arr.size else float('nan')
        mean_L_bern = float(z_crit * mean_sqrt_u_bern_global)
        rho_ci_length = (float(mean_L_rem / mean_L_bern)
                         if (np.isfinite(mean_L_bern) and mean_L_bern > 0) else float('nan'))

        summary[s] = {
            'n_accepted': n_acc,
            'true_var_uncond': float(true_var_tau),
            'true_var_cond': float(true_var_cond),
            'mean_u_hat_n': float(np.mean(u_arr)),
            'mean_u_hat_n_bern': mean_u_hat_n_bern_global,
            'mean_v_hat_n': float(np.mean(v_arr)),
            'mean_ratio': float(np.mean(ratios)),
            'conserv_uncond': float(np.mean(u_arr) / true_var_tau) if true_var_tau > 0 else float('nan'),
            'conserv_cond': float(np.mean(v_arr) / true_var_cond) if true_var_cond > 0 else float('nan'),
            'coverage_uncond': float(np.mean(r['ci_covers_uncond'])),
            'coverage_cond': float(np.mean(r['ci_covers_cond'])),
            'v_p': float(sd['v_p']), 'p_cov': int(sd['p_cov']),
            'R_squared': float(sd['R_squared']),
            'theoretical_ratio': float(1 - (1 - sd['v_p']) * sd['R_squared']),

            'mean_L_rem': mean_L_rem,
            'mean_L_bern': mean_L_bern,
            'rho_ci_length': rho_ci_length,
        }

        if s == 'all' and n_acc > 0:
            v_savje_arr = np.array(r['v_hat_n_savje'])
            v_savje_valid = v_savje_arr[~np.isnan(v_savje_arr)]
            if len(v_savje_valid) > 0 and true_var_cond > 0:
                summary[s]['conserv_cond_savje'] = float(np.mean(v_savje_valid) / true_var_cond)
                summary[s]['mean_v_hat_n_savje'] = float(np.mean(v_savje_valid))
            v_uncen_arr = np.array(r['v_hat_n_uncen'])
            v_uncen_valid = v_uncen_arr[~np.isnan(v_uncen_arr)]
            if len(v_uncen_valid) > 0 and true_var_cond > 0:
                summary[s]['conserv_cond_uncen'] = float(np.mean(v_uncen_valid) / true_var_cond)
                summary[s]['mean_v_hat_n_uncen'] = float(np.mean(v_uncen_valid))
    return summary


def run_cv_experiment(cfg, target_log_ratios=None, total_signal_std=30.0, n_jobs=-1):

    if target_log_ratios is None:
        target_log_ratios = np.linspace(-4, 4, 9)

    shared = create_shared_experiment(cfg)
    d_local = compute_local_dep_degrees(shared['adj'])
    lambda_max = compute_lambda_max(shared['adj'])
    print(f"Outcome Type {cfg.outcome_type} | Rerand covs: {cfg.rerand_covs}")
    print(f"Block stds: static={shared['std_pre']:.4f}, dynamic={shared['std_trt']:.4f}")
    print(f"d_i: mean={np.mean(d_local):.1f}, max={np.max(d_local):.0f} | lambda_max={lambda_max:.1f}")
    print(f"Running {len(target_log_ratios)} sweep points...\n")

    sweep = Parallel(n_jobs=n_jobs, verbose=5)(
        delayed(_run_single_cv)(cfg, shared, total_signal_std, float(d), cfg.seed + 1000 + i)
        for i, d in enumerate(target_log_ratios)
    )
    sweep.sort(key=lambda x: x['target_log_ratio'])

    results = {
        'config': asdict(cfg),
        'model': {'outcome_type': cfg.outcome_type, 'description': shared['description']},
        'block_stds': {'std_pre': shared['std_pre'], 'std_trt': shared['std_trt']},
        'total_signal_std': total_signal_std,
        'd_local_mean': float(np.mean(d_local)),
        'lambda_max': lambda_max,
        'sweep': sweep,
        'timestamp': datetime.now().isoformat(),
    }

    for s in sweep:
        a = s.get('all', {})
        if isinstance(a, dict) and 'rho_ci_length' in a:
            print(f"  log2(R)={s['actual_log_ratio']:6.2f} | "
                  f"rho_CIlen={a['rho_ci_length']:.3f} (var-ratio={a.get('mean_ratio', float('nan')):.3f}), "
                  f"R2={a.get('R_squared',0):.3f}, cov_cond={a['coverage_cond']:.3f}")
    return results


def plot_ratio(results, save_path=None):


    setup_plot_style()
    sweep = results['sweep']
    ratios_x = [s['actual_log_ratio'] for s in sweep]

    fig, ax = plt.subplots(figsize=(12, 9))
    for strat, (color, marker, label) in STRATEGY_STYLES.items():
        y = []
        for s in sweep:
            sd = s.get(strat, {})
            if isinstance(sd, dict):
                rho = sd.get('rho_ci_length', np.nan)
                if rho is None or not np.isfinite(rho):


                    v = sd.get('mean_v_hat_n', np.nan)
                    u = sd.get('mean_u_hat_n_bern', sd.get('mean_u_hat_n', np.nan))
                    rho = np.sqrt(v / u) if (np.isfinite(v) and np.isfinite(u) and u > 0) else np.nan
                y.append(rho)
            else:
                y.append(np.nan)
        ax.plot(ratios_x, y, f'{marker}-', color=color, lw=2.5, label=label, ms=10)

    ax.set_xlabel(XLABEL_LOG_RATIO, labelpad=12)
    ax.set_ylabel(r'$\rho = \widehat{L}_{\mathrm{ReM}} \,/\, \widehat{L}_{\mathrm{Bern}}$', labelpad=12)
    ax.axhline(1.0, color='gray', ls=':', alpha=0.7)
    ax.legend(frameon=True, loc='best', fontsize=16)
    ax.grid(True, ls='--', alpha=0.5)

    plt.subplots_adjust(top=0.90, bottom=0.12, left=0.12, right=0.95)
    if save_path: plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


def plot_coverage(results, save_path=None):

    setup_plot_style()
    sweep = results['sweep']
    ratios_x = [s['actual_log_ratio'] for s in sweep]

    fig, axes = plt.subplots(1, 3, figsize=(20, 7), sharey=True)
    for ax, strat, (_, _, title) in zip(axes, ['static', 'dynamic', 'all'], STRATEGY_STYLES.values()):
        cu = [s.get(strat, {}).get('coverage_uncond', np.nan) for s in sweep]
        cc = [s.get(strat, {}).get('coverage_cond', np.nan) for s in sweep]
        ax.plot(ratios_x, cu, 'o-', color='#1f77b4', lw=2, label='Unconditional CI', ms=7)
        ax.plot(ratios_x, cc, 's-', color='#d62728', lw=2, label='Conditional CI', ms=7)
        ax.axhline(1 - results['config']['ci_alpha'], color='gray', ls='--', alpha=0.7, label='Nominal (95\%)')
        ax.set_xlabel(XLABEL_LOG_RATIO)
        ax.set_title(title)
        ax.legend(fontsize=14, frameon=True)
        ax.grid(True, ls='--', alpha=0.5)
        ax.set_ylim(0.8, 1.05)
    axes[0].set_ylabel('Coverage Probability')

    plt.subplots_adjust(top=0.90, bottom=0.14, left=0.06, right=0.98, wspace=0.12)
    if save_path: plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


def plot_conservativeness(results, save_path=None):

    setup_plot_style()
    sweep = results['sweep']
    ratios_x = [s['actual_log_ratio'] for s in sweep]

    fig, axes = plt.subplots(1, 3, figsize=(20, 7), sharey=True)
    for ax, strat, (_, _, title) in zip(axes, ['static', 'dynamic', 'all'], STRATEGY_STYLES.values()):
        cu = [s.get(strat, {}).get('conserv_uncond', np.nan) for s in sweep]
        cc = [s.get(strat, {}).get('conserv_cond', np.nan) for s in sweep]
        ax.plot(ratios_x, cu, 'o-', color='#1f77b4', lw=2,
                label=r'$\mathrm{E}[\hat{u}_n] \,/\, u_n$', ms=7)
        ax.plot(ratios_x, cc, 's-', color='#d62728', lw=2,
                label=r'$\mathrm{E}[\hat{v}_n] \,/\, v_n$', ms=7)
        ax.axhline(1.0, color='gray', ls='--', alpha=0.7, label='Conservative threshold')
        ax.set_xlabel(XLABEL_LOG_RATIO)
        ax.set_title(title)
        ax.legend(fontsize=14, frameon=True)
        ax.grid(True, ls='--', alpha=0.5)
    axes[0].set_ylabel(r'Conservativeness Ratio')

    plt.subplots_adjust(top=0.90, bottom=0.14, left=0.06, right=0.98, wspace=0.12)
    if save_path: plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


def plot_comparison(results, save_path=None):


    setup_plot_style()
    sweep = results['sweep']
    ratios_x = [s['actual_log_ratio'] for s in sweep]

    def _safe_log(x):
        return np.log(x) if (x is not None and np.isfinite(x) and x > 0) else np.nan

    ours_y, uncen_y, lmax_y = [], [], []
    for s in sweep:
        a = s.get('all', {})
        if not isinstance(a, dict):
            ours_y.append(np.nan); uncen_y.append(np.nan); lmax_y.append(np.nan); continue
        ours_y.append(_safe_log(a.get('conserv_cond', np.nan)))
        uncen_y.append(_safe_log(a.get('conserv_cond_uncen', np.nan)))
        lmax_y.append(_safe_log(a.get('conserv_cond_savje', np.nan)))

    fig, ax = plt.subplots(figsize=(12, 9))

    ax.plot(ratios_x, ours_y, 'o-', color='#d62728', lw=3.0,
            label=r'ours: group cent. + per-unit $d_i$', ms=10, alpha=0.9)
    ax.plot(ratios_x, uncen_y, '^-.', color='#ff7f0e', lw=3.0,
            label=r'no centering + per-unit $d_i$', ms=10, alpha=0.9)
    ax.plot(ratios_x, lmax_y, 's--', color='#9467bd', lw=3.0,
            label=r'group cent. + $\lambda_{\max}$', ms=10, alpha=0.9)
    ax.axhline(0.0, color='gray', ls=':', alpha=0.8, lw=1.5)
    ax.set_xlabel(XLABEL_LOG_RATIO, labelpad=12)
    ax.set_ylabel(r'$\mathrm{CR}$', labelpad=12)
    ax.legend(frameon=True, framealpha=0.95, fancybox=True, loc='best', fontsize=16)
    ax.grid(True, ls='--', alpha=0.6)

    plt.subplots_adjust(top=0.90, bottom=0.12, left=0.12, right=0.95)
    if save_path: plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


def plot_all(results, save_dir=None):
    prefix = f"cv_type{results['model']['outcome_type']}_"


    plot_ratio(results, save_path=f"{save_dir}/{prefix}ratio.pdf" if save_dir else None)
    plot_coverage(results, save_path=f"{save_dir}/{prefix}coverage.pdf" if save_dir else None)
    plot_conservativeness(results, save_path=f"{save_dir}/{prefix}conserv.pdf" if save_dir else None)
    plot_comparison(results, save_path=f"{save_dir}/{prefix}comparison.pdf" if save_dir else None)


print('Loaded: run_conservative_var')


In [ ]:
import os, traceback

EXPERIMENTS = [
    (1, ['neighbor_prop']),
    (1, ['neighbor_weighted_X']),
    (1, ['neighbor_prop', 'neighbor_weighted_X']),
    (2, ['neighbor_prop']),
    (2, ['neighbor_weighted_X']),
    (2, ['neighbor_prop', 'neighbor_weighted_X']),
    (3, ['neighbor_prop']),
    (3, ['neighbor_weighted_X']),
    (3, ['neighbor_prop', 'neighbor_weighted_X']),
    (5, ['neighbor_prop']),
    (5, ['neighbor_weighted_X']),
    (5, ['neighbor_prop', 'neighbor_weighted_X']),
    (6, ['neighbor_prop']),
    (6, ['neighbor_weighted_X']),
    (6, ['neighbor_prop', 'neighbor_weighted_X']),
    (7, ['neighbor_prop']),
    (7, ['neighbor_weighted_X']),
    (7, ['neighbor_prop', 'neighbor_weighted_X']),
    (8, ['neighbor_prop']),
    (8, ['neighbor_weighted_X']),
    (8, ['neighbor_prop', 'neighbor_weighted_X']),
]

_COVS_SHORT = {
    'neighbor_prop': 'np',
    'neighbor_weighted_X': 'nwx',
    'second_order_neighbor_weighted_X': 'sonwx',
}
def covs_short(covs):
    return '+'.join(_COVS_SHORT.get(c, c) for c in covs)

os.makedirs('results_cv_data', exist_ok=True)
os.makedirs('results_cv_picture', exist_ok=True)
LOG_PATH = 'results_cv_data/cv_batch_log.txt'
with open(LOG_PATH, 'w') as logf:
    logf.write(f'Batch CV started: {datetime.now().isoformat()}\n')
    logf.write(f'Total experiments: {len(EXPERIMENTS)}\n\n')

for i, (otype, covs) in enumerate(EXPERIMENTS, 1):
    sn = covs_short(covs)
    tag = f'type{otype}_covs-{sn}'
    print(f'\n[{i}/{len(EXPERIMENTS)}] {tag} | covs={covs}')
    t0 = datetime.now()
    try:
        cfg = Config(
            n_units=500,
            outcome_type=otype,
            n_pre=6,
            n_hidden=1,
            rerand_covs=covs,
            acceptance_rate=0.05,
            n_accepted=2000,
            n_mc=5000,
            p_treat=0.5,
            ci_alpha=0.05,


            graph_type='ba',
            graph_m=5,


        )
        results = run_cv_experiment(
            cfg,
            target_log_ratios=np.linspace(-4, 4, 9),
            total_signal_std=30.0,
            n_jobs=1,
        )
        save_results(results, save_dir='results_cv_data', prefix=f'cv_covs-{sn}_')
        base = f'results_cv_picture/cv_covs-{sn}_type{otype}'


        plot_ratio(results, save_path=f'{base}_ratio.pdf'); plt.close('all')
        plot_coverage(results, save_path=f'{base}_coverage.pdf'); plt.close('all')
        plot_conservativeness(results, save_path=f'{base}_conserv.pdf'); plt.close('all')
        plot_comparison(results, save_path=f'{base}_comparison.pdf'); plt.close('all')
        dt = (datetime.now() - t0).total_seconds()
        status = f'OK ({dt:.0f}s) -> {base}_*.pdf'
    except Exception as e:
        status = f'FAILED: {e!r}'
        traceback.print_exc()
    print(f'  [{tag}] {status}')
    with open(LOG_PATH, 'a') as logf:
        logf.write(f'[{datetime.now().isoformat()}] {tag} | {status}\n')

print('\n=== Batch CV complete ===')
with open(LOG_PATH, 'a') as logf:
    logf.write(f'\nBatch CV complete: {datetime.now().isoformat()}\n')
